#### Import Libraries and General Settings

In [ ]:
import sys
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import (
    KFold,
    RepeatedKFold,
    train_test_split,
    learning_curve,
    cross_validate,
)
from sklearn.ensemble import HistGradientBoostingRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import optuna.logging

optuna.logging.set_verbosity(optuna.logging.WARNING)
optuna.logging.set_verbosity(optuna.logging.ERROR)
import logging

logging.getLogger("lightgbm").setLevel(logging.ERROR)
import warnings

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)


import catboost, xgboost, lightgbm, sklearn, matplotlib

print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("Optuna:", optuna.__version__)
print("CatBoost:", catboost.__version__)
print("XGBoost:", xgboost.__version__)
print("LightGBM:", lightgbm.__version__)
print("Sklearn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)

#### Load dataset

In [ ]:
melbourne_file_path = 'dataset/melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path) 

#### Dataset

In [ ]:
melbourne_data

#### Statistical summary of the columns

In [ ]:
melbourne_data.describe()

#### Number of rows, columns, data types, number of non-null values

In [ ]:
melbourne_data.info()

#### Copy the dataset

In [ ]:
melbourne_data_copy = melbourne_data.copy(deep=True)

#### Copy the Suburb column

In [ ]:
suburb_copy = melbourne_data['Suburb'].copy(deep=True)

#### Data processing

In [ ]:
# Convert strings to integers
for c in melbourne_data.columns:
    if melbourne_data[c].dtype == "str":
        lbl = LabelEncoder()
        lbl.fit(list(melbourne_data[c].values))
        melbourne_data[c] = lbl.transform(list(melbourne_data[c].values))

LabelEncoder was used to transform strings into integers and encode categorical variables. Among several strategies tested, this encoding provided the best performance in cross-validation with the models, without introducing bias into the predictions and maintaining the interpretability of the model.

#### Complete missing data

In [ ]:
melbourne_data.fillna({"Car": 0}, inplace=True)
melbourne_data["YearBuilt"] = melbourne_data.groupby("Postcode")["YearBuilt"].transform(
    lambda x: x.fillna(x.mean())
)
melbourne_data["BuildingArea"] = melbourne_data.groupby("Postcode")[
    "BuildingArea"
].transform(lambda x: x.fillna(x.mean()))
melbourne_data["CouncilArea"] = melbourne_data.groupby("Postcode")[
    "CouncilArea"
].transform(lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x)

#### Delete rows where the missing YearBuilt and BuildingArea values ​​do not match any other Postcode

In [ ]:
melbourne_data = melbourne_data.dropna(subset=["YearBuilt"])
melbourne_data = melbourne_data.dropna(subset=["BuildingArea"])

#### Create new features to improve the accuracy rate

In [ ]:
melbourne_data["Rooms_type"] = (
    melbourne_data["Rooms"] * melbourne_data["Type"]
)
melbourne_data["Bedroom2_type"] = (
    melbourne_data["Bedroom2"] * melbourne_data["Type"]
) 

#### Separate the dataset into features and price

In [ ]:
X = melbourne_data.drop(['Price'], axis=1)
y = melbourne_data['Price']

#### Split train/test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)


*Train–Test Split Strategy*

**A 90%–10% train–test split was used to maximize training data, with cross-validation employed to ensure robust and stable evaluation despite the smaller test set.**
 Hyperparameter selection and intermediate model evaluation were performed using cross-validation, which compensates for the smaller proportion of the test set and ensures robust and stable performance metrics.


#### CV evaluation function

In [ ]:
def evaluate_model_cv(model, X, y, cv_splits=5):
    cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
    scoring = {"rmse": "neg_root_mean_squared_error", "r2": "r2"}

    cv_results = cross_validate(
        model, X, y, cv=cv, scoring=scoring, return_train_score=True, n_jobs=1
    )

    train_rmse = -cv_results["train_rmse"]
    test_rmse = -cv_results["test_rmse"]
    train_r2 = cv_results["train_r2"]
    test_r2 = cv_results["test_r2"]

    return {
        "Train RMSE": np.mean(train_rmse),
        "Test RMSE": np.mean(test_rmse),
        "Test RMSE std": np.std(test_rmse),
        "Gap RMSE": np.mean(test_rmse) - np.mean(train_rmse),
        "Gap RMSE %": (np.mean(test_rmse) - np.mean(train_rmse))
        / np.mean(test_rmse)
        * 100,
        "Train R2": np.mean(train_r2),
        "Test R2": np.mean(test_r2),
        "Gap R2": (np.mean(train_r2) - np.mean(test_r2)),
    }

#### Define ranges for each parameter of each model
*Optuna was used to optimize each model*

*Selected models:*
- **CatBoostRegressor**
- **XGBRegressor**
- **LGBMRegressor**
- **HistGradientBoostingRegressor**

In [ ]:
# -------------------------
# Objective CatBoostRegressor
# -------------------------
def objective_cat(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.06),
        "depth": trial.suggest_int("depth", 3, 6),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 80, 200),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 20, 80, log=True),
        "rsm": trial.suggest_float("rsm", 0.65, 0.85),
        "subsample": trial.suggest_float("subsample", 0.6, 0.8),
        "random_strength": trial.suggest_float("random_strength", 3, 8),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 1.6, 2.4),
        "loss_function": "RMSE",
        "early_stopping_rounds": 50,
        "verbose": 0,
    }
    model = CatBoostRegressor(**params)
    res = evaluate_model_cv(model, X, y)
    return res["Test RMSE"]


# -------------------------
# Objective XGBRegressor
# -------------------------
def objective_xgb(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.06),
        "max_depth": trial.suggest_int("max_depth", 1, 3),
        "min_child_weight": trial.suggest_int("min_child_weight", 10, 40),
        "subsample": trial.suggest_float("subsample", 0.5, 0.8),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.8),
        "gamma": trial.suggest_float("gamma", 0.5, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 50.0, log=True),
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
    }
    model = XGBRegressor(**params)
    res = evaluate_model_cv(model, X, y)
    return res["Test RMSE"]


# -------------------------
# Objective LGBMRegressor
# -------------------------
def objective_lgb(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 5, 15),
        "max_depth": trial.suggest_int("max_depth", 30, 35),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 2.5, 3.4),
        "reg_lambda": trial.suggest_float("reg_lambda", 2.5, 3.4),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 35, 42),
        "min_child_samples": 60,
        "boosting_type": "gbdt",
        "objective": "regression",
        "metric": "rmse",
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
        "force_row_wise": True,
    }
    model = LGBMRegressor(**params)
    res = evaluate_model_cv(model, X, y)
    return res["Test RMSE"]


# -------------------------
# Objective HistGradientBoostingRegressor
# -------------------------
def objective_hgb(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.06),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "max_iter": trial.suggest_int("max_iter", 100, 200),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 80, 300),
        "l2_regularization": trial.suggest_float(
            "l2_regularization", 0.5, 50.0, log=True
        ),
        "random_state": 42,
    }
    model = HistGradientBoostingRegressor(**params)
    res = evaluate_model_cv(model, X, y)
    return res["Test RMSE"]

#### Create studies and optimize

In [ ]:
studies = {}
objectives = {
    "Ctbr": objective_cat,
    "Lgbr": objective_lgb,
    "Xgbr": objective_xgb,
    "Hgbr": objective_hgb,
}
n_trials = 2

for name, obj in objectives.items():
    print(f"\nOptimizing {name} ...")
    study = optuna.create_study(
        direction="minimize",
        study_name=f"{name}_rmse",
        sampler=optuna.samplers.TPESampler(multivariate=True, group=True, seed=42),
    )
    study.optimize(obj, n_trials=n_trials, show_progress_bar=True)
    studies[name] = study
    print(f"Best params: {name}")
    for k, v in study.best_trial.params.items():
        print(f"  {k}: {v}")

#### Extract final metrics from each best trial

In [ ]:
results_list = []

for name, study in studies.items():
    best_trial = study.best_trial
    if name == "Ctbr":
        model = CatBoostRegressor(**best_trial.params, random_seed=42, verbose=0)
    elif name == "Lgbr":
        model = LGBMRegressor(**best_trial.params, random_state=42, n_jobs=-1)
    elif name == "Xgbr":
        model = XGBRegressor(
            **best_trial.params,
            random_state=42,
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=-1
        )
    else:
        model = HistGradientBoostingRegressor(**best_trial.params, random_state=42)

    res = evaluate_model_cv(model, X, y)
    res["Model"] = name
    results_list.append(res)

#### Display metrics

In [ ]:
results_df = pd.DataFrame(results_list)
results_df = results_df[
    [
        "Model",
        "Train RMSE",
        "Test RMSE",
        "Test RMSE std",
        "Gap RMSE",
        "Gap RMSE %",
        "Train R2",
        "Test R2",
        "Gap R2",
    ]
]

cols_highlight_min = ["Test RMSE", "Gap RMSE", "Gap R2", "Gap RMSE %"]
cols_highlight_max = ["Test R2"]

styled_df = (
    results_df.style.highlight_min(subset=cols_highlight_min, color="#00441b")
    .highlight_max(subset=cols_highlight_max, color="#00441b")
    .format(
        {
            "Gap R2": "{:.4f}",
            "Gap RMSE": "{:.4f}",
            "Gap RMSE %": "{:.4f}",
            "Test RMSE": "{:.4f}",
            "Test R2": "{:.4f}",
        }
    )
)
styled_df

|             |                  |         |
|-------------|------------------|---------|
|Test RMSE    |(lower = better)  | 🏆 XGBR |
|Gap RMSE     |(lower = better)  | 🏆 HGBR |
|Gap RMSE (%) |(lower = better)  | 🏆 HGBR |
|Test R²      |(higher = better) | 🏆 XGBR |
|Gap R2       |(lower = better)  | 🏆 HGBR |

*Model Comparison and Selection*

**XGBRegressor, LightGBM, and CatBoostRegressor achieve the best results in terms of predictive performance**, with XGBRegressor standing out due to its lower RMSE and higher coefficient of determination on the test set. However, this model exhibits a larger gap between training and validation performance, indicating a higher degree of overfitting.

**CatBoost and LightGBM provide a more balanced trade-off between accuracy and generalization**, showing moderate train–test gaps.

**Although HistGradientBoostingRegressor shows slightly lower predictive performance**, it presents the smallest gap between training and test results, suggesting more stable and robust behavior.  

**Therefore, HistGradientBoostingRegressor is selected as the final model based on its superior generalization performance and stability.**



#### Final graphs with RMSE test, Gap % and R² test

In [ ]:
sns.set(style="whitegrid")

x = np.arange(len(results_df))
width = 0.25

fig, ax1 = plt.subplots(figsize=(10, 6))

bar1 = ax1.bar(
    x - width / 2, results_df["Test RMSE"], width, label="Test RMSE", color="skyblue"
)

bar2 = ax1.bar(
    x + width / 2,
    results_df["Gap RMSE %"] * 1000,
    width,
    label="Gap RMSE % (x1000)",
    color="salmon",
    alpha=0.7,
)

ax1.set_xlabel("Modelo")
ax1.set_xticks(x)
ax1.set_xticklabels(results_df["Model"])
ax1.set_ylabel("RMSE / Gap x1000")
ax1.set_title("Benchmark Modelos: Test RMSE vs Gap % vs Test R²")
ax1.legend(loc="center left")

ax2 = ax1.twinx()
ax2.plot(x, results_df["Test R2"], color="green", marker="o", label="Test R²")
ax2.set_ylabel("Test R²")
ax2.set_ylim(0, 1)
ax2.legend(loc="center right")

for rect in bar1:
    height = rect.get_height()
    ax1.text(
        rect.get_x() + rect.get_width() / 2.0,
        height + 5000,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

for rect in bar2:
    height = rect.get_height()
    ax1.text(
        rect.get_x() + rect.get_width() / 2.0,
        height + 5000,
        f"{height/1000:.1f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

#### Instantiate the selected model

In [ ]:
hgbr = HistGradientBoostingRegressor(**studies['Hgbr'].best_trial.params, random_state=42)

#### Plot Learning curve

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    estimator=hgbr,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

train_rmse = -train_scores
val_rmse = -val_scores

train_mean = train_rmse.mean(axis=1)
train_std = train_rmse.std(axis=1)

val_mean = val_rmse.mean(axis=1)
val_std = val_rmse.std(axis=1)

plt.figure(figsize=(9, 6))

plt.plot(train_sizes, train_mean, "o-", color="tab:blue", label="Train RMSE")
plt.plot(train_sizes, val_mean, "o-", color="tab:orange", label="Validation RMSE")

plt.fill_between(
    train_sizes,
    train_mean - train_std,
    train_mean + train_std,
    alpha=0.15,
    color="tab:blue",
)

plt.fill_between(
    train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color="tab:orange"
)

plt.xlabel("Number of training samples")
plt.ylabel("RMSE")
plt.title("Learning Curve - HistGradientBoostingRegressor")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#### Train the selected model and obtain predictions

In [ ]:
hgbr.fit(X_train, y_train)
y_pred = hgbr.predict(X_test)

#### Get all housing prices by suburb and number of rooms

In [ ]:
prices_df = melbourne_data_copy.groupby(["Suburb", "Rooms"], as_index=False).agg(
    Minimum_price=("Price", "min"),
    Maximum_price=("Price", "max"),
    Average_price=("Price", "mean"),
)

#### Display real prices, predictions, and comparisons of similar homes by suburb and number of rooms.

In [ ]:
prediction_df = pd.DataFrame()
prediction_df["Sale price"] = y_test.astype(int)
prediction_df["Predicted price"] = y_pred.astype(int)
prediction_df["Suburb"] = suburb_copy
prediction_df["Rooms"] = X.Rooms
merge_df = pd.merge(prediction_df, prices_df, on=["Suburb", "Rooms"], how="inner")
merge_df["Average_price"] = merge_df["Average_price"].astype(int)
merge_df["Minimum_price"] = merge_df["Minimum_price"].astype(int)
merge_df["Maximum_price"] = merge_df["Maximum_price"].astype(int)
merge_df = merge_df.map(lambda x: f"{x:,}" if isinstance(x, (int, float)) else x)
merge_df.sample(10)